## Decision Tree Classifier

In [1]:
import pandas as pd
import numpy as np

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint
from sklearn.metrics import roc_auc_score

In [2]:
heart_test = pd.read_csv("Data/heart_test.csv")
heart_train = pd.read_csv("Data/heart_train.csv")
rice_test = pd.read_csv("Data/rice_test.csv")
rice_train = pd.read_csv("Data/rice_train.csv")
mushrooms_test = pd.read_csv("Data/mushrooms_test.csv")
mushrooms_test = pd.read_csv("Data/mushrooms_train.csv")
wine_test = pd.read_csv("Data/wine_test.csv")
wine_train = pd.read_csv("Data/wine_train.csv")

In [3]:
heart_test.head()

,age,gender,impluse,pressurehight,pressurelow,glucose,kcm,troponin,class
0,-0.601832,0.725420,-0.119600,0.870505,1.618292,-0.618424,0.029144,-0.312368,1
1,-0.891897,0.725420,0.253002,-0.206378,-0.362251,0.758037,-0.279361,-0.318727,0
2,-1.979638,0.725420,-0.051854,1.139726,1.476825,1.075682,6.752612,-0.321112,1
3,-0.021704,0.725420,-0.153472,-0.552519,-1.211055,-0.552248,-0.300666,-0.321112,0
4,0.775974,-1.378511,-0.017981,-0.014077,-0.786653,-0.552248,-0.162935,-0.063563,1


In [4]:
param_dist = {
    'ccp_alpha': uniform(0, 1),          
    'max_depth': randint(1, 30),        
    'min_samples_leaf': randint(1, 60), 
    'min_samples_split': randint(2, 60) 
}

In [ ]:
datasets = {
    "heart": ("Data/heart_train.csv", "Data/heart_test.csv"),
    "rice": ("Data/rice_train.csv", "Data/rice_test.csv"),
    "mushrooms": ("Data/mushrooms_train.csv", "Data/mushrooms_test.csv"),
    "wine": ("Data/wine_train.csv", "Data/wine_test.csv")
}

param_dist = {
    'ccp_alpha': uniform(0, 1),
    'max_depth': randint(1, 30),
    'min_samples_leaf': randint(1, 60),
    'min_samples_split': randint(2, 60)
}

all_results = []

for name, (train_path, test_path) in datasets.items():

    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)

    X_train, y_train = train.iloc[:, :-1], train.iloc[:, -1]
    X_test, y_test = test.iloc[:, :-1], test.iloc[:, -1]

    dt = DecisionTreeClassifier(random_state=42)
    random_search = RandomizedSearchCV(
        estimator=dt,
        param_distributions=param_dist,
        n_iter=100,
        scoring='roc_auc',
        cv=3,
        random_state=42,
        n_jobs=1,
        return_train_score=False
    )
    random_search.fit(X_train, y_train)

    cv_results = pd.DataFrame(random_search.cv_results_)

    for i, params in enumerate(random_search.cv_results_['params']):
        model = DecisionTreeClassifier(random_state=42, **params)
        model.fit(X_train, y_train)
        y_proba = model.predict_proba(X_test)[:, 1]
        test_auc = roc_auc_score(y_test, y_proba)

        all_results.append({
            "dataset": name,
            "params": params,
            "cv_roc_auc": cv_results.loc[i, 'mean_test_score'],
            "test_roc_auc": test_auc
        })

results_df = pd.DataFrame(all_results)

In [6]:
results_df

,dataset,params,cv_roc_auc,test_roc_auc
0,heart,"{'ccp_alpha': 0.3745401188473625, 'max_depth':...",0.500000,0.500000
1,heart,"{'ccp_alpha': 0.7796910002727693, 'max_depth':...",0.500000,0.500000
2,heart,"{'ccp_alpha': 0.15599452033620265, 'max_depth'...",0.953421,0.975272
3,heart,"{'ccp_alpha': 0.33370861113902184, 'max_depth'...",0.500000,0.500000
4,heart,"{'ccp_alpha': 0.020584494295802447, 'max_depth...",0.990549,0.975272
...,...,...,...,...
395,wine,"{'ccp_alpha': 0.29529058841893874, 'max_depth'...",0.500000,0.500000
396,wine,"{'ccp_alpha': 0.697015740995268, 'max_depth': ...",0.500000,0.500000
397,wine,"{'ccp_alpha': 0.5528199769079077, 'max_depth':...",0.500000,0.500000
398,wine,"{'ccp_alpha': 0.8101133946791808, 'max_depth':...",0.500000,0.500000


In [7]:
results_df.to_csv("results_decisiontree.csv", index=False)

In [8]:
best_per_dataset = (
    results_df.sort_values(by=["dataset", "test_roc_auc"], ascending=[True, False]).groupby("dataset", as_index=False).first()
)

In [9]:
best_per_dataset

,dataset,params,cv_roc_auc,test_roc_auc
0,heart,"{'ccp_alpha': 0.15599452033620265, 'max_depth'...",0.953421,0.975272
1,mushrooms,"{'ccp_alpha': 0.005061583846218687, 'max_depth...",0.998251,0.999580
2,rice,"{'ccp_alpha': 0.3745401188473625, 'max_depth':...",0.983241,0.986596
3,wine,"{'ccp_alpha': 0.005061583846218687, 'max_depth...",0.977226,0.986781


In [10]:
params_df = best_per_dataset["params"].apply(pd.Series)

In [11]:
params_df

,ccp_alpha,max_depth,min_samples_leaf,min_samples_split
0,0.155995,11.0,11.0,25.0
1,0.005062,14.0,18.0,3.0
2,0.374540,29.0,15.0,44.0
3,0.005062,14.0,18.0,3.0


In [12]:
mean_params = params_df.mean()
mean_params

ccp_alpha             0.135164
max_depth            17.000000
min_samples_leaf     15.500000
min_samples_split    18.750000
dtype: float64

In [13]:
mean_params_dict = mean_params.to_dict()
for param in ["max_depth", "min_samples_leaf", "min_samples_split"]:
    mean_params_dict[param] = int(round(mean_params_dict[param]))
mean_results = []

for name, (train_path, test_path) in datasets.items():
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)

    X_train, y_train = train.iloc[:, :-1], train.iloc[:, -1]
    X_test, y_test = test.iloc[:, :-1], test.iloc[:, -1]

    model = DecisionTreeClassifier(random_state=42, **mean_params_dict)
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_test)[:, 1]
    mean_auc = roc_auc_score(y_test, y_proba)

    mean_results.append({
        "dataset": name,
        "mean_test_roc_auc": mean_auc
    })

mean_df = pd.DataFrame(mean_results)

In [14]:
mean_df

,dataset,mean_test_roc_auc
0,heart,0.975272
1,rice,0.986596
2,mushrooms,0.888848
3,wine,0.874031


In [15]:
results_df = results_df.merge(mean_df, on="dataset")
results_df["diff_from_mean"] = results_df["mean_test_roc_auc"] - results_df["test_roc_auc"]
results_df

,dataset,params,cv_roc_auc,test_roc_auc,mean_test_roc_auc,diff_from_mean
0,heart,"{'ccp_alpha': 0.3745401188473625, 'max_depth':...",0.500000,0.500000,0.975272,0.475272
1,heart,"{'ccp_alpha': 0.7796910002727693, 'max_depth':...",0.500000,0.500000,0.975272,0.475272
2,heart,"{'ccp_alpha': 0.15599452033620265, 'max_depth'...",0.953421,0.975272,0.975272,0.000000
3,heart,"{'ccp_alpha': 0.33370861113902184, 'max_depth'...",0.500000,0.500000,0.975272,0.475272
4,heart,"{'ccp_alpha': 0.020584494295802447, 'max_depth...",0.990549,0.975272,0.975272,0.000000
...,...,...,...,...,...,...
395,wine,"{'ccp_alpha': 0.29529058841893874, 'max_depth'...",0.500000,0.500000,0.874031,0.374031
396,wine,"{'ccp_alpha': 0.697015740995268, 'max_depth': ...",0.500000,0.500000,0.874031,0.374031
397,wine,"{'ccp_alpha': 0.5528199769079077, 'max_depth':...",0.500000,0.500000,0.874031,0.374031
398,wine,"{'ccp_alpha': 0.8101133946791808, 'max_depth':...",0.500000,0.500000,0.874031,0.374031


In [16]:
results_df.sort_values(by="diff_from_mean",ascending=True).head(20)

,dataset,params,cv_roc_auc,test_roc_auc,mean_test_roc_auc,diff_from_mean
375,wine,"{'ccp_alpha': 0.005061583846218687, 'max_depth...",0.977226,0.986781,0.874031,-0.112750
275,mushrooms,"{'ccp_alpha': 0.005061583846218687, 'max_depth...",0.998251,0.999580,0.888848,-0.110732
361,wine,"{'ccp_alpha': 0.016587828927856152, 'max_depth...",0.975313,0.983351,0.874031,-0.109320
304,wine,"{'ccp_alpha': 0.020584494295802447, 'max_depth...",0.973858,0.983351,0.874031,-0.109320
261,mushrooms,"{'ccp_alpha': 0.016587828927856152, 'max_depth...",0.972570,0.984168,0.888848,-0.095320
204,mushrooms,"{'ccp_alpha': 0.020584494295802447, 'max_depth...",0.947909,0.947025,0.888848,-0.058178
253,mushrooms,"{'ccp_alpha': 0.05147875124998935, 'max_depth'...",0.947909,0.947025,0.888848,-0.058178
211,mushrooms,"{'ccp_alpha': 0.06505159298527952, 'max_depth'...",0.947909,0.947025,0.888848,-0.058178
299,mushrooms,"{'ccp_alpha': 0.08159418040024036, 'max_depth'...",0.947909,0.947025,0.888848,-0.058178
239,mushrooms,"{'ccp_alpha': 0.07697990982879299, 'max_depth'...",0.947909,0.947025,0.888848,-0.058178


In [17]:
results_df.to_csv("results_decisiontree.csv", index=False)

In [19]:
#TEST DLA INNEGO ZBIORU

# from sklearn.preprocessing import OneHotEncoder, StandardScaler
# from sklearn.model_selection import train_test_split
# def scale_dataframes(train_df, test_df, target_column):
  
#     numeric_cols = [col for col in train_df.columns if col != target_column]
    
#     scaler = StandardScaler()
#     scaler.fit(train_df[numeric_cols])
    
#     train_scaled = train_df.copy()
#     test_scaled = test_df.copy()
    
#     train_scaled[numeric_cols] = scaler.transform(train_df[numeric_cols])
#     test_scaled[numeric_cols] = scaler.transform(test_df[numeric_cols])
    
#     return train_scaled, test_scaled

# blood_df = pd.read_csv('Data/blood.csv')
# blood_df.head()
# blood_df['class'] = blood_df['Class'].map({1: 0, 2: 1})
# blood_train_df, blood_test_df = train_test_split(blood_df, test_size=0.25, stratify=blood_df['class'], random_state=42)
# blood_train_df, blood_test_df = scale_dataframes(blood_train_df, blood_test_df, target_column='class')
# blood_train_df.to_csv('Data/blood_train.csv', index=False)
# blood_test_df.to_csv('Data/blood_test.csv', index=False)